# Rag Meal Prep

In [95]:
import json
import os
from langchain.document_loaders.base import BaseLoader
from langchain.schema.output_parser import StrOutputParser
from langchain.schema import Document

In [74]:
# Load environment variables (for API keys)
load_dotenv(find_dotenv(), override=True)
api_key = os.environ.get("OPENAI_API_KEY")
uri_key = os.environ.get("ZILLIZ_URI")
token_key=os.environ.get("ZILLIZ_TOKEN")

## Classe per caricamento ricette

In [75]:
class RecipeJSONLoader(BaseLoader):
    """Caricatore personalizzato per file JSON di ricette."""
    
    def __init__(self, file_path):
        self.file_path = file_path
    
    def load(self):
        """Carica le ricette dal file JSON."""
        with open(self.file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        documents = []
        for recipe in data["recipes"]:
            # Formattazione del testo della ricetta
            recipe_text = f"Title: {recipe['title']}\n"
            recipe_text += f"Cuisine: {recipe['cuisine']}\n"
            recipe_text += f"Meal Type: {recipe['mealType']}\n"
            recipe_text += f"Preparation Time: {recipe['prepTime']}\n"
            recipe_text += f"Cooking Time: {recipe['cookTime']}\n"
            recipe_text += f"Servings: {recipe['servings']}\n"
            recipe_text += f"Dietary Info: {', '.join(recipe['dietaryInfo'])}\n\n"
            
            recipe_text += f"Ingredients:\n"
            for ingredient in recipe['ingredients']:
                recipe_text += f"- {ingredient}\n"
            
            recipe_text += f"\nInstructions:\n{recipe['instructions']}"
            
            # Metadati specifici della ricetta
            metadata = {
                "id": recipe["id"],
                "title": recipe["title"],
                "cuisine": recipe["cuisine"],
                "mealType": recipe["mealType"],
                "source": self.file_path
            }
            
            # Creazione del documento
            doc = Document(page_content=recipe_text, metadata=metadata)
            documents.append(doc)
        
        return documents

In [76]:
def load_recipes_with_langchain(file_path):
    """Funzione aggiornata per caricare le ricette."""
    loader = RecipeJSONLoader(file_path)
    return loader.load()


In [77]:
# Chunking dei documenti
def create_chunks(documents, chunk_size=1000, chunk_overlap=200):
    from langchain.text_splitter import RecursiveCharacterTextSplitter
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        is_separator_regex=False,
    )
    
    chunks = text_splitter.split_documents(documents)
    print(f"Creati {len(chunks)} chunks da {len(documents)} documenti")
    return chunks

In [78]:
# Creazione e caricamento del database vettoriale su Zilliz/Milvus
def create_vector_store(chunks, connection_args):
    from langchain_huggingface import HuggingFaceEmbeddings
    from langchain_community.vectorstores import Milvus
    
    # Inizializzazione del modello di embedding
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    
    # Creazione del database vettoriale
    vector_store = Milvus.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name="recipes_collection",
        connection_args=connection_args,
        drop_old=True  # Attenzione: questo eliminerà una collezione esistente con lo stesso nome
    )
    
    return vector_store

In [79]:
# 5. Ricerca di ricette simili
def search_recipes(vector_store, query, top_k=5):
    # Ricerca semantica
    results = vector_store.similarity_search_with_score(query, k=top_k)
    
    print(f"\nRisultati per la query: '{query}'")
    for i, (doc, score) in enumerate(results):
        print(f"\nRisultato {i+1} (Similarità: {1/(1+score):.4f}):")
        print(f"Titolo: {doc.metadata.get('title')}")
        print(f"Cucina: {doc.metadata.get('cuisine')}")
        print(f"Tipo pasto: {doc.metadata.get('mealType')}")
        print(f"Contenuto: {doc.page_content[:200]}...")


## Running Code

In [80]:
# Connessione a Zilliz

connection_args = {
    "uri": uri_key,
    "token": token_key,
    "secure": True
}

In [81]:
file_path = "recipe_data/new_recipes.json"
# Caricamento delle ricette
documents = load_recipes_with_langchain(file_path)

In [82]:
#Creazione dei chunks
chunks = create_chunks(documents)

Creati 34 chunks da 20 documenti


In [83]:
 # 3. Visualizzazione di un esempio di chunk
print("\nEsempio di chunk:")
print(f"Metadati: {chunks[0].metadata}")
print('-'*50)
print(f"Contenuto: {chunks[0].page_content[:300]}...")


Esempio di chunk:
Metadati: {'id': 11, 'title': 'Lasagne con Ragù', 'cuisine': 'Italian', 'mealType': 'Dinner', 'source': 'recipe_data/new_recipes.json'}
--------------------------------------------------
Contenuto: Title: Lasagne con Ragù
Cuisine: Italian
Meal Type: Dinner
Preparation Time: 45 minutes
Cooking Time: 1 hour 40 minutes
Servings: 8
Dietary Info: High protein, Contains gluten, Contains dairy, Contains eggs

Ingredients:
- 500g lasagne sheets
- 500g ground beef
- 200g ground pork
- 1 onion, finely c...


In [84]:
#Creazione del database vettoriale
vector_store = create_vector_store(chunks, connection_args)

In [110]:
search_recipes(vector_store, "Lasagne al ragù", top_k=2)


Risultati per la query: 'Lasagne al ragù'

Risultato 1 (Similarità: 0.5181):
Titolo: Lasagne con Ragù
Cucina: Italian
Tipo pasto: Dinner
Contenuto: Title: Lasagne con Ragù
Cuisine: Italian
Meal Type: Dinner
Preparation Time: 45 minutes
Cooking Time: 1 hour 40 minutes
Servings: 8
Dietary Info: High protein, Contains gluten, Contains dairy, Contain...

Risultato 2 (Similarità: 0.4383):
Titolo: Lasagne con Ragù
Cucina: Italian
Tipo pasto: Dinner
Contenuto: Instructions:
Preheat oven to 375°F. For the ragù, heat oil and brown the meat. Add onion, carrots, celery, and garlic. Cook until softened. Add wine and simmer until reduced. Add tomatoes, paste, her...


In [96]:
# Initialize the language model
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# Create a retriever from the vector store
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# Define a template for generating responses
template = """
You are a helpful cooking assistant that provides information about recipes and meal planning.
Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context:
{context}

Question: {question}

Answer:
"""

# Create a prompt from the template
prompt = ChatPromptTemplate.from_template(template)

# Define a function to format documents
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

# Build the RAG chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [99]:
response1 = rag_chain.invoke("Che ingredienti per preparare le lasegne?")

print(f"Query: Che ingredienti per preparare le lasegne?")
print(f"Response:\n{response1}\n")

Query: Che ingredienti per preparare le lasegne?
Response:
- 500g fogli di lasagne
- 500g carne macinata di manzo
- 200g carne macinata di maiale
- 1 cipolla, tritata finemente
- 2 carote, tagliate a dadini
- 2 gambi di sedano, tagliati a dadini
- 3 spicchi d'aglio, tritati
- 800g pomodori pelati
- 2 cucchiai di concentrato di pomodoro
- 1 tazza di vino rosso
- 1 cucchiaino di origano secco
- 1 cucchiaino di basilico secco
- 500g ricotta
- 300g mozzarella grattugiata
- 100g Parmigiano grattugiato
- 2 uova
- Sale e pepe q.b.
- 2 cucchiai di olio d'oliva
- Basilico fresco per guarnire



In [104]:
print(documents[0])

page_content='Title: Lasagne con Ragù
Cuisine: Italian
Meal Type: Dinner
Preparation Time: 45 minutes
Cooking Time: 1 hour 40 minutes
Servings: 8
Dietary Info: High protein, Contains gluten, Contains dairy, Contains eggs

Ingredients:
- 500g lasagne sheets
- 500g ground beef
- 200g ground pork
- 1 onion, finely chopped
- 2 carrots, diced
- 2 celery stalks, diced
- 3 garlic cloves, minced
- 800g crushed tomatoes
- 2 tbsp tomato paste
- 1 cup red wine
- 1 tsp dried oregano
- 1 tsp dried basil
- 500g ricotta cheese
- 300g mozzarella, grated
- 100g Parmesan cheese, grated
- 2 eggs
- Salt and pepper to taste
- 2 tbsp olive oil
- Fresh basil for garnish

Instructions:
Preheat oven to 375°F. For the ragù, heat oil and brown the meat. Add onion, carrots, celery, and garlic. Cook until softened. Add wine and simmer until reduced. Add tomatoes, paste, herbs, salt, and pepper. Simmer for 1 hour. Mix ricotta, half the mozzarella, half the Parmesan, eggs, salt, and pepper. In a baking dish, layer s

In [105]:
## Adding MMemory
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain

# Initialize memory
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

# Create a conversational retrieval chain
conversation_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    verbose=True
)

/var/folders/tl/zb5lzmw1379d6f96zdj440lm0000gn/T/ipykernel_59529/102649236.py:6: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(


In [106]:
# Test conversation with memory
print("Starting conversation with memory...\n")

# First query
query1 = "What ingredients do I need for a vegetable stir fry?"
result1 = conversation_chain.invoke({"question": query1})
print(f"Query: {query1}")
print(f"Response: {result1['answer']}\n")

# Follow-up query that relies on conversation context
query2 = "How long does it take to cook?"
result2 = conversation_chain.invoke({"question": query2})
print(f"Query: {query2}")
print(f"Response: {result2['answer']}\n")

# Another follow-up query
query3 = "Can I make it vegan?"
result3 = conversation_chain.invoke({"question": query3})
print(f"Query: {query3}")
print(f"Response: {result3['answer']}")

Starting conversation with memory...



> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
System: Use the following pieces of context to answer the user's question. 
If you don't know the answer, just say that you don't know, don't try to make up an answer.
----------------
Instructions:
Heat olive oil in a large pot. Add onion, carrots, celery, and garlic, cook until softened. Add tomato paste and cook for 1 minute. Add lentils, potatoes, crushed tomatoes, bay leaf, rosemary, and broth. Bring to a boil, then simmer for 20 minutes. Add pasta and cook until al dente, about 10 minutes more, adding more broth if needed. The result should be a thick soup. Remove bay leaf and rosemary sprig. Season with salt and pepper. Serve with a drizzle of olive oil and grated Parmesan.

Instructions:
Heat 1 tbsp olive oil in a large pot. Add onions and cook until translucent. Add garlic and cook for 1 minute. Add bell pepper and cook for 5 minutes. 

In [123]:
## FInalmente il Meal Planner
# Define a specialized prompt for meal planning
meal_planner_template = """
You are a helpful meal planning assistant. Based on the recipe information provided and the user's preferences,
create a meal plan as requested. Use only the recipes mentioned in the context or variations of them.

Provide a detailed meal plan with:
-recipe suggestions (for every meal preparation tips from instructions)
-modifications needed to meet the user's requirements.

Format your response in a clear, organized way with headings and bullet points as appropriate.

Context (Recipe Information):
{context}

User Request: {question}


"""

meal_planner_prompt = ChatPromptTemplate.from_template(meal_planner_template)

# Build the meal planner RAG chain
meal_planner_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | meal_planner_prompt
    | llm
    | StrOutputParser()
)

In [127]:
# Test the meal planner with a complex request
meal_plan_request = """I need a week low carb meal plan with breakfast, lunch, and dinner.
I prefer high-protein options and I have these ingredients available: salmon, tofu, eggs, yogurt,
various vegetables, and fruits. I don't like mushrooms."""

meal_plan = meal_planner_chain.invoke(meal_plan_request)

print("Meal Plan Request:")
print(meal_plan_request)
print("\nGenerated Meal Plan:")
print(meal_plan)

Meal Plan Request:
I need a week low carb meal plan with breakfast, lunch, and dinner.
I prefer high-protein options and I have these ingredients available: salmon, tofu, eggs, yogurt,
various vegetables, and fruits. I don't like mushrooms.

Generated Meal Plan:
**Low Carb Meal Plan for a Week**

**Day 1:**

**Breakfast:**
- Scrambled Eggs with Spinach and Bell Peppers
  - Saute spinach and bell peppers in olive oil.
  - Add beaten eggs and cook until scrambled.
  - Season with salt and pepper.

**Lunch:**
- Buddha Bowl con Ceci e Quinoa (from the provided recipe)
  - Follow the recipe instructions, omitting mushrooms.
  - Use tofu instead of chickpeas for a high-protein option.

**Dinner:**
- Grilled Salmon with Roasted Vegetables
  - Marinate salmon with olive oil, lemon juice, and herbs.
  - Grill until cooked through.
  - Roast a variety of vegetables like zucchini, bell peppers, and asparagus with olive oil, salt, and pepper.

**Day 2:**

**Breakfast:**
- Greek Yogurt Parfait with

## lista della spesa

In [141]:
## Lista della spesa

# Definire il template per la generazione della lista della spesa
shopping_list_template = """
Sei un assistente specializzato nella creazione di liste della spesa. Basandoti sul piano alimentare fornito,
crea una lista della spesa completa e organizzata.

Analizza attentamente il piano alimentare seguente e identifica tutti gli ingredienti necessari:

{meal_plan}

Genera una lista della spesa organizzata per categorie (es. Proteine, Verdure, Frutta, Latticini, Condimenti, ecc.).
Raggruppa gli ingredienti simili e indica la quantità approssimativa necessaria per coprire il periodo del piano alimentare.
Considera che alcune ricette possono utilizzare gli stessi ingredienti.

Formatta la risposta in modo chiaro e ordinato, utilizzando elenchi puntati per facilitare l'uso durante la spesa.
"""

# Creare un prompt dal template
shopping_list_prompt = ChatPromptTemplate.from_template(shopping_list_template)

# Costruire il chain per la lista della spesa
shopping_list_chain = (
    {"meal_plan": RunnablePassthrough()}
    | shopping_list_prompt
    | llm
    | StrOutputParser()
)

# Test della funzionalità della lista della spesa
# Utilizziamo il piano alimentare già generato in precedenza
shopping_list = shopping_list_chain.invoke(meal_plan)

print("Lista della Spesa generata dal Piano Alimentare:")
print(shopping_list)

Lista della Spesa generata dal Piano Alimentare:
**Proteine:**
- Eggs (1 dozen)
- Tofu (2 blocks)
- Salmon fillets (4)
- Grilled chicken strips (1 lb)
- Shrimp (1 lb)

**Verdure:**
- Spinach (1 bag)
- Bell peppers (6)
- Zucchini (6)
- Asparagus (1 bunch)
- Broccoli (1 head)
- Snap peas (1 cup)
- Avocado (4)
- Cauliflower (1 head)
- Lettuce (1 head)
- Cucumber (2)
- Tomatoes (4)
- Onions (2)
- Mixed vegetables for stir-fry (1 bag)

**Frutta:**
- Berries (mixed) (2 cups)
- Lemons (4)

**Cereali e Legumi:**
- Quinoa (2 cups)
- Chickpeas (1 can)
- Chia seeds (1 cup)

**Latticini:**
- Greek yogurt (1 large container)
- Feta cheese (1 block)

**Condimenti e Altri:**
- Olive oil
- Soy sauce
- Lemon juice
- Herbs (e.g. basil, parsley)
- Salt
- Pepper
- Nuts (e.g. almonds)
- Pesto sauce
- Chimichurri sauce
- Almond milk
- Tahini

Questa lista della spesa dovrebbe coprire gli ingredienti necessari per seguire il piano alimentare per una settimana. Assicurati di controllare la dispensa per eventu